<a href="https://colab.research.google.com/github/vaibhavchavhan45/langchain-core-components/blob/main/Re_rankers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Re-Ranker** (using .predict method)

# What Re-Ranker Expects
Get the data from Retriver + User Query --> Make pairs of data(retriever's chunk) and query --> Assigns scores to chunk on the basis of user query --> Sort the chunks (descending order) on basis of scores --> Filters the best chunks which matches immensly with query --> Pass to the LLM

Re-Ranker sits between the retriver and LLM

In [ ]:
!pip install sentence-transformers

  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.9.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Us

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
# Instance of re-ranker
re_ranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

In [ ]:
query = "What is Retrival Augmented Generation?"

In [ ]:
chunks = [
     "RAG is a system that combines retrieval and generation using external knowledge.",
    "RAG allows LLMs to answer questions using retrieved documents.",
    "Vector databases store embeddings for similarity search.",
    "Fine-tuning modifies model weights using labeled data.",
     "Transformers are neural network models used in NLP.",
]

In [ ]:
# make pairs as query and chunk
pairs = [(query, chunk) for chunk in chunks]

In [ ]:
# Calculate the scores
scores = re_ranker.predict(pairs)

In [ ]:
scores

array([ -9.24999 , -11.470181, -11.461213, -11.452642, -11.343206],
      dtype=float32)

In [ ]:
# combine chunks and scores in a array      E.g. [('first chunk', score)]
ranked_chunks = list(zip(chunks, scores))

In [ ]:
ranked_chunks

[('RAG is a system that combines retrieval and generation using external knowledge.',
  np.float32(-9.24999)),
 ('Transformers are neural network models used in NLP.',
  np.float32(-11.343206)),
 ('Fine-tuning modifies model weights using labeled data.',
  np.float32(-11.452642)),
 ('Vector databases store embeddings for similarity search.',
  np.float32(-11.461213)),
 ('RAG allows LLMs to answer questions using retrieved documents.',
  np.float32(-11.470181))]

In [ ]:
# sort the chunks in descending order using score as a parameter
ranked_chunks.sort(key=lambda x: x[1], reverse=True)

In [ ]:
ranked_chunks

[('RAG is a system that combines retrieval and generation using external knowledge.',
  np.float32(-9.24999)),
 ('Transformers are neural network models used in NLP.',
  np.float32(-11.343206)),
 ('Fine-tuning modifies model weights using labeled data.',
  np.float32(-11.452642)),
 ('Vector databases store embeddings for similarity search.',
  np.float32(-11.461213)),
 ('RAG allows LLMs to answer questions using retrieved documents.',
  np.float32(-11.470181))]

In [ ]:
# retriving the score and chunk
for chunk, score in ranked_chunks:
    print(f"Score: {score:.4f} | Chunk: {chunk}")

Score: -9.2500 | Chunk: RAG is a system that combines retrieval and generation using external knowledge.
Score: -11.3432 | Chunk: Transformers are neural network models used in NLP.
Score: -11.4526 | Chunk: Fine-tuning modifies model weights using labeled data.
Score: -11.4612 | Chunk: Vector databases store embeddings for similarity search.
Score: -11.4702 | Chunk: RAG allows LLMs to answer questions using retrieved documents.


**FLOW using .predict :**
1. User query + Chunks or docs
2. Make pairs of query and chunk
3. called .predict() to generate scores
4. Combines chunks and generated scores in a list
5. Sorted that list on the basis of score
6. Extract the top results

# **Re-Ranker** (using .rank method)

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
reRanker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
query = "What is Retrieval Augmented Generation?"

In [ ]:
docs = [
    "RAG is a system that combines retrieval and generation using external knowledge.",
    "RAG allows LLMs to answer questions using retrieved documents.",
    "Vector databases store embeddings for similarity search.",
    "Fine-tuning modifies model weights using labeled data.",
     "Transformers are neural network models used in NLP.",
]

In [ ]:
results = reRanker.rank(query, docs)

In [ ]:
results

[{'corpus_id': 0, 'score': np.float32(0.22187115)},
 {'corpus_id': 1, 'score': np.float32(-10.252987)},
 {'corpus_id': 4, 'score': np.float32(-11.34815)},
 {'corpus_id': 2, 'score': np.float32(-11.411005)},
 {'corpus_id': 3, 'score': np.float32(-11.461946)}]

In [ ]:
for result in results:
    score = result['score']
    item = chunks[result['corpus_id']]
    print(f"Score: {score:.4f} | Item: {item}")

Score: 0.2219 | Item: RAG is a system that combines retrieval and generation using external knowledge.
Score: -10.2530 | Item: RAG allows LLMs to answer questions using retrieved documents.
Score: -11.3482 | Item: Transformers are neural network models used in NLP.
Score: -11.4110 | Item: Vector databases store embeddings for similarity search.
Score: -11.4619 | Item: Fine-tuning modifies model weights using labeled data.


**FLOW of Re-Ranker usig .rank()**
1. User query + Chunks or docs
2. Called **.rank**(query, chunks) directly (Got sorted results automatically)
3. Extract the result